# Stock Market Prediction Notebook

        This notebook is the cleaned analytical companion to the Flask project. It presents the project in a structured way for review, starting from data loading and exploratory analysis, then moving into feature engineering, model training, evaluation, and short-horizon forecasting.


## Project Objective

        The goal is to forecast the next trading day's S&P 500 closing price using historical OHLCV market data:

        - Open
        - High
        - Low
        - Close
        - Adjusted Close
        - Volume

        This notebook keeps the workflow readable and presentation-friendly, while the Flask application packages the same logic into a recruiter-ready interface.


In [ ]:
from pathlib import Path

        import matplotlib.pyplot as plt
        import numpy as np
        import pandas as pd
        import seaborn as sns
        from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
        from sklearn.metrics import mean_absolute_error, mean_squared_error
        from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler

        sns.set_theme(style="whitegrid")
        plt.rcParams["figure.figsize"] = (12, 5)
        plt.rcParams["axes.titlesize"] = 14


In [ ]:
DATA_PATH = Path(r"C:\Users\Administrator\Desktop\PROJECT2\S&P dataset.csv")

        df = pd.read_csv(DATA_PATH)
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.sort_values("Date").reset_index(drop=True)

        print(f"Dataset shape: {df.shape}")
        print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
        df.head()


## Data Overview

        The dataset contains more than two decades of index history. Before building any model, it is useful to inspect data quality, trend behavior, and the relationship between price and volume.


In [ ]:
df.describe().T


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

        axes[0].plot(df["Date"], df["Close"], color="#0b7285", linewidth=2)
        axes[0].set_title("S&P 500 Closing Price Over Time")
        axes[0].set_ylabel("Close")

        axes[1].plot(df["Date"], df["Volume"], color="#c92a2a", linewidth=1.5)
        axes[1].set_title("Trading Volume Over Time")
        axes[1].set_ylabel("Volume")
        axes[1].set_xlabel("Date")

        plt.tight_layout()
        plt.show()


In [ ]:
correlation_frame = df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].corr()

        plt.figure(figsize=(8, 6))
        sns.heatmap(correlation_frame, annot=True, cmap="YlGnBu", fmt=".2f")
        plt.title("Feature Correlation Heatmap")
        plt.show()


## Feature Engineering

        Raw OHLCV values are useful, but time-series models usually improve when we add lag features, moving averages, short-term returns, momentum, and volatility indicators.


In [ ]:
featured = df.copy()

        featured["return_1d"] = featured["Close"].pct_change()
        featured["return_5d"] = featured["Close"].pct_change(5)
        featured["volume_change"] = featured["Volume"].pct_change()
        featured["high_low_spread"] = (featured["High"] - featured["Low"]) / featured["Close"]
        featured["open_close_spread"] = (featured["Close"] - featured["Open"]) / featured["Open"]
        featured["ma_5"] = featured["Close"].rolling(5).mean()
        featured["ma_10"] = featured["Close"].rolling(10).mean()
        featured["ma_20"] = featured["Close"].rolling(20).mean()
        featured["volatility_10"] = featured["return_1d"].rolling(10).std()
        featured["momentum_10"] = featured["Close"] - featured["Close"].shift(10)

        for lag in range(1, 6):
            featured[f"close_lag_{lag}"] = featured["Close"].shift(lag)
            featured[f"volume_lag_{lag}"] = featured["Volume"].shift(lag)

        featured["target_close"] = featured["Close"].shift(-1)
        featured = featured.dropna().reset_index(drop=True)

        print(featured.shape)
        featured.head()


In [ ]:
feature_columns = [
            "Open", "High", "Low", "Close", "Adj Close", "Volume",
            "return_1d", "return_5d", "volume_change",
            "high_low_spread", "open_close_spread",
            "ma_5", "ma_10", "ma_20",
            "volatility_10", "momentum_10",
            "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5",
            "volume_lag_1", "volume_lag_2", "volume_lag_3", "volume_lag_4", "volume_lag_5",
        ]

        X = featured[feature_columns]
        y = featured["target_close"]

        split_index = int(len(featured) * 0.8)
        X_train = X.iloc[:split_index]
        X_test = X.iloc[split_index:]
        y_train = y.iloc[:split_index]
        y_test = y.iloc[split_index:]
        previous_close = featured["Close"].iloc[split_index:]

        print("Train shape:", X_train.shape, y_train.shape)
        print("Test shape:", X_test.shape, y_test.shape)


## Model Training

        Instead of keeping many half-working deep learning experiments in the notebook, this cleaned version uses a stable and explainable forecasting workflow:

        - Time-series aware validation with `TimeSeriesSplit`
        - Hyperparameter tuning with `GridSearchCV`
        - Multiple candidate regressors
        - Final selection based on validation performance


In [ ]:
candidates = {
            "random_forest": (
                Pipeline([
                    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
                ]),
                {
                    "model__n_estimators": [150, 250],
                    "model__max_depth": [8, 14],
                    "model__min_samples_split": [2, 5],
                },
            ),
            "gradient_boosting": (
                Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", GradientBoostingRegressor(random_state=42)),
                ]),
                {
                    "model__n_estimators": [120, 180],
                    "model__learning_rate": [0.03, 0.05],
                    "model__max_depth": [2, 3],
                },
            ),
        }

        splitter = TimeSeriesSplit(n_splits=4)
        searches = {}

        for name, (pipeline, params) in candidates.items():
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=params,
                scoring="neg_mean_absolute_percentage_error",
                cv=splitter,
                n_jobs=-1,
            )
            search.fit(X_train, y_train)
            searches[name] = search
            print(name, search.best_score_, search.best_params_)


In [ ]:
best_name = max(searches, key=lambda name: searches[name].best_score_)
        best_search = searches[best_name]
        best_model = best_search.best_estimator_

        predictions = best_model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mae = mean_absolute_error(y_test, predictions)
        mape = np.mean(np.abs((y_test - predictions) / y_test)) * 100
        direction_accuracy = (
            np.sign(y_test.values - previous_close.values) ==
            np.sign(predictions - previous_close.values)
        ).mean() * 100

        metrics = pd.Series({
            "Best model": best_name,
            "RMSE": round(rmse, 2),
            "MAE": round(mae, 2),
            "MAPE (%)": round(mape, 2),
            "Directional accuracy (%)": round(direction_accuracy, 2),
        })

        metrics


In [ ]:
comparison = pd.DataFrame({
            "Date": featured["Date"].iloc[split_index:].dt.strftime("%Y-%m-%d").values,
            "Actual Close": y_test.values,
            "Predicted Close": predictions,
        })

        comparison.tail(12)


In [ ]:
plt.figure(figsize=(14, 6))
        plt.plot(comparison["Date"].tail(90), comparison["Actual Close"].tail(90), label="Actual", color="#0b7285", linewidth=2)
        plt.plot(comparison["Date"].tail(90), comparison["Predicted Close"].tail(90), label="Predicted", color="#c92a2a", linestyle="--", linewidth=2)
        plt.title("Actual vs Predicted Closing Prices")
        plt.xlabel("Date")
        plt.ylabel("Close")
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()
        plt.show()


## Short-Horizon Forecast

        To support the Flask dashboard, the project also produces a short recursive forecast for the next few business days.


In [ ]:
forecast_frame = featured.copy()
        future_rows = []

        for _ in range(5):
            latest = forecast_frame.iloc[-1:].copy()
            x_input = latest[feature_columns]
            predicted_close = float(best_model.predict(x_input)[0])

            next_date = latest["Date"].iloc[0] + pd.offsets.BDay(1)
            base_close = float(latest["Close"].iloc[0])
            base_volume = float(latest["Volume"].iloc[0])

            synthetic_row = {
                "Date": next_date,
                "Open": base_close,
                "High": max(base_close, predicted_close) * 1.01,
                "Low": min(base_close, predicted_close) * 0.99,
                "Close": predicted_close,
                "Adj Close": predicted_close,
                "Volume": base_volume,
            }

            source = pd.concat([
                forecast_frame[["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]],
                pd.DataFrame([synthetic_row])
            ], ignore_index=True)

            featured_source = source.copy()
            featured_source["return_1d"] = featured_source["Close"].pct_change()
            featured_source["return_5d"] = featured_source["Close"].pct_change(5)
            featured_source["volume_change"] = featured_source["Volume"].pct_change()
            featured_source["high_low_spread"] = (featured_source["High"] - featured_source["Low"]) / featured_source["Close"]
            featured_source["open_close_spread"] = (featured_source["Close"] - featured_source["Open"]) / featured_source["Open"]
            featured_source["ma_5"] = featured_source["Close"].rolling(5).mean()
            featured_source["ma_10"] = featured_source["Close"].rolling(10).mean()
            featured_source["ma_20"] = featured_source["Close"].rolling(20).mean()
            featured_source["volatility_10"] = featured_source["return_1d"].rolling(10).std()
            featured_source["momentum_10"] = featured_source["Close"] - featured_source["Close"].shift(10)

            for lag in range(1, 6):
                featured_source[f"close_lag_{lag}"] = featured_source["Close"].shift(lag)
                featured_source[f"volume_lag_{lag}"] = featured_source["Volume"].shift(lag)

            featured_source["target_close"] = featured_source["Close"].shift(-1)
            forecast_frame = featured_source.dropna().reset_index(drop=True)

            future_rows.append({
                "Date": next_date.strftime("%Y-%m-%d"),
                "Predicted Close": round(predicted_close, 2),
            })

        pd.DataFrame(future_rows)


## Final Notes

        This cleaned notebook is designed to support discussion during interviews:

        - It uses clear file paths instead of Colab-only paths.
        - It removes broken cells and noisy experiments.
        - It focuses on explainable forecasting, feature engineering, and model evaluation.
        - It aligns with the Flask application shown in the project folder.

        If needed later, a true LSTM implementation can be added as an advanced extension, but this notebook is already much more presentable and reliable than the original draft.
